# Trabajo Práctico Integrador - Introducción al Análisis de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> La empresa busca comprender qué factores influyen en la cancelación de reservas en hoteles urbanos y resort, analizando variables como fechas de estadía, tipo de cliente, canal de reserva, historial previo y tarifas, con el fin de identificar patrones y mejorar la gestión operativa.


## Desarrollado por

- **Apellido y nombre:** Matias Carro y Hugo Catalan
- **Comisión:** 11
- **Dataset asignado:** Dataset K
- **Entrega:** Primer Entrega - Semana 3
- **Fecha:** 

## Importación de librerías

### Librerias a utilizar:

- **pandas:** para cargar el dataset y trabajar con datos en forma de tablas (DataFrames).
- **numpy:** para realizar operaciones numéricas y manejar arreglos de forma eficiente.



In [3]:
import pandas as pd
import numpy as np

print("Versión de Pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)
print("\nLibrerías cargadas correctamente.")


Versión de Pandas: 3.0.5
Versión de numpy: 2.5.1

Librerías cargadas correctamente.


## 1. Presentacion del problema

El caso de estudio se centra en el análisis de reservas hoteleras con el objetivo de comprender qué factores están asociados a la cancelación de estadías. El dataset asignado contiene información detallada de cada reserva, incluyendo tipo de hotel, fechas de llegada, duración de la estadía, composición del grupo, país de origen, canal de reserva, tipo de cliente, historial previo, tarifa promedio por noche y características operativas como depósito, agente, pedidos especiales y cambios realizados.


### Relación entre datos, información y conocimiento
En este trabajo partimos de los **datos** que son valores crudos del sistema de reservas: fechas, cantidades, categorías y códigos.  
Mediante el análisis exploratorio estos datos se convierten en **información**, como distribuciones, patrones y diferencias entre reservas canceladas y no canceladas.  
A partir de esa información generamos el **conocimiento** que nos permite entender el comportamiento de los clientes y detectar factores que podrían influir en la cancelación de una reserva.  

Esta relación es clave para el caso: los datos del hotel por sí solos no dicen nada, pero al transformarlos en información y luego interpretarlos, podemos identificar variables relevantes (como `lead_time`, `deposit_type` o `customer_type`) que ayudan a explicar por qué algunas reservas se cancelan y otras no.

### Ciclo de vida del análisis
Este trabajo se enmarca en el ciclo de vida del análisis de datos, que incluye:
1. Obtención del dataset asignado.  
2. Comprensión inicial del problema (cancelaciones hoteleras).  
3. Exploración y limpieza mínima (EDA).  
4. Transformación y preparación de variables relevantes.  
5. Interpretación y comunicación de resultados.

### Variable objetivo
La **variable objetivo** del análisis es **`is_canceled`**, que indica si la reserva fue cancelada (`1`) o no (`0`).  
Su distribución será calculada y analizada en las próximas secciones para comprender el comportamiento general del conjunto de datos y orientar las preguntas del análisis.

### Preguntas iniciales que orientan el trabajo
- ¿Qué características diferencian a las reservas canceladas de las no canceladas?  
- ¿Influyen el tipo de hotel o el canal de reserva en la cancelación?  
- ¿Las reservas con mayor anticipación (`lead_time`) presentan mayor probabilidad de cancelación?  
- ¿Los clientes con pedidos especiales o estacionamiento tienden a cancelar menos?  
- ¿Las políticas de depósito (`deposit_type`) reducen la cancelación?  
- ¿Existen segmentos de mercado con mayor riesgo de cancelación?  

## 2. Carga del dataset

Se carga el Dataset perteneciente a la comisión 11:


In [4]:
df = pd.read_csv("hotel booking TPI grupo K.csv")

print("Dataset cargado correctamente.")
print("\nVista de los primeros datos: ")
df.head()


Dataset cargado correctamente.

Vista de los primeros datos: 


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-014269,Resort Hotel,0,17,2025,February,8,21,2025-02-21,0,...,E,0,No Deposit,NaN,292.0,0,Transient,58.00,1,1
1,HB-018087,Resort Hotel,0,6,2023,November,44,1,2023-11-01,2,...,D,3,No Deposit,240.0,NaN,0,Transient,58.00,0,2
2,HB-022870,Resort Hotel,0,45,2024,April,15,8,2024-04-08,0,...,D,1,No Deposit,240.0,NaN,0,Transient-Party,65.00,0,2
3,HB-048154,City Hotel,0,95,2024,March,11,17,2024-03-17,2,...,A,0,No Deposit,9.0,NaN,0,Transient,73.95,0,1
4,HB-060351,City Hotel,1,277,2024,November,45,7,2024-11-07,1,...,A,0,Non Refund,NaN,NaN,0,Transient,100.00,0,0


## 4. Estructura general:

Filas y Columnas:

In [5]:
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.info())


El dataset tiene 25000 filas y 32 columnas.
Columnas del dataset:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                          

El dataset tiene 25.000 filas y 32 columnas.  
Las variables incluyen información temporal, categórica y numérica relevante para el análisis.


### Diccionario de variables 

| Variable | Traducción | Representación | Tipo de Datos |
|----------|------------|----------------|-----------|
| booking_id | ID de reserva | Identificador único de cada reserva. | str |
| hotel | Tipo de hotel | Indica si la reserva corresponde a City Hotel (Ciudad) o Resort Hotel (Resort). | str |
| is_canceled | Cancelada | Indica si la reserva fue cancelada (1) o no (0). | int64 |
| lead_time | Anticipación | Días entre la fecha de reserva y la fecha de llegada. | int64 |
| arrival_date_year | Año de llegada | Año en el que el huésped llega al hotel. | int64 |
| arrival_date_month | Mes de llegada | Mes en el que el huésped llega al hotel. | str |
| arrival_date_week_number | Semana de llegada | Número de semana del año en la que llega el huésped. | int64 |
| arrival_date_day_of_month | Día del mes de llegada | Día del mes en el que llega el huésped. | int64 |
| arrival_date | Fecha de llegada | Fecha completa de llegada (YYYY-MM-DD). | str |
| stays_in_weekend_nights | Noches de fin de semana | Cantidad de noches en fines de semana. | int64 |
| stays_in_week_nights | Noches de semana | Cantidad de noches de lunes a jueves. | int64 |
| adults | Adultos | Número de adultos en la reserva. | int64 |
| children | Niños | Número de niños en la reserva. | float64 |
| babies | Bebés | Número de bebés en la reserva. | int64 |
| meal | Tipo de comida | Plan de comidas asociado a la reserva (BB, HB, SC, etc.). | str |
| country | País | País de origen del huésped. | str |
| market_segment | Segmento de mercado | Tipo de cliente según el canal de adquisición. | str |
| distribution_channel | Canal de distribución | Canal por el cual se realizó la reserva. | str |
| is_repeated_guest | Huésped repetido | Indica si el cliente ya se alojó anteriormente. | int64 |
| previous_cancellations | Cancelaciones previas | Cantidad de reservas previas canceladas por el cliente. | int64 |
| previous_bookings_not_canceled | Reservas previas no canceladas | Cantidad de reservas previas completadas por el cliente. | int64 |
| reserved_room_type | Habitación reservada | Tipo de habitación solicitada originalmente. | str |
| assigned_room_type | Habitación asignada | Tipo de habitación finalmente asignada. | str |
| booking_changes | Cambios en la reserva | Número de modificaciones realizadas a la reserva. | int64 |
| deposit_type | Tipo de depósito | Política de depósito aplicada. | str |
| agent | Agente | Código del agente que gestionó la reserva. | float64 |
| company | Compañía | Código de la empresa asociada a la reserva. | float64 |
| days_in_waiting_list | Días en lista de espera | Tiempo que la reserva permaneció en espera antes de confirmarse. | int64 |
| customer_type | Tipo de cliente | Clasificación del cliente (Transient, Contract, Group, etc.). | str |
| adr | Tarifa promedio diaria | Precio promedio por noche de la reserva. | float64 |
| required_car_parking_spaces | Estacionamiento requerido | Cantidad de espacios de estacionamiento solicitados. | int64 |
| total_of_special_requests | Pedidos especiales | Número de solicitudes especiales realizadas por el cliente. | int64 |



## Tipos de variables 

Clasificacion de las variables por su tipo de datos

In [6]:
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cadenas = df.select_dtypes(include=['str']).columns.tolist()

print("Cantidad de variables numéricas:", len(numericas))
print("Cantidad de variables de texto / fecha:", len(cadenas))



Cantidad de variables numéricas: 20
Cantidad de variables de texto / fecha: 12


### Clasificación inicial de las variables

| Variable | Tipo | Representación |
|----------|------|----------------|
| booking_id | Categórica (ID) | Identificador único de reserva. |
| hotel | Categórica | Tipo de hotel. |
| is_canceled | Numérica (booleana) | Indica si la reserva fue cancelada. |
| lead_time | Numérica | Días de anticipación de la reserva. |
| arrival_date_year | Numérica | Año de llegada. |
| arrival_date_month | Categórica | Mes de llegada. |
| arrival_date_week_number | Numérica | Semana del año. |
| arrival_date_day_of_month | Numérica | Día del mes. |
| arrival_date | Temporal | Fecha completa de llegada. |
| stays_in_weekend_nights | Numérica | Noches de fin de semana. |
| stays_in_week_nights | Numérica | Noches de semana. |
| adults | Numérica | Cantidad de adultos. |
| children | Numérica | Cantidad de niños. |
| babies | Numérica | Cantidad de bebés. |
| meal | Categórica | Tipo de comida. |
| country | Categórica | País de origen. |
| market_segment | Categórica | Segmento de mercado. |
| distribution_channel | Categórica | Canal de distribución. |
| is_repeated_guest | Numérica (booleana) | Indica si el huésped ya se alojó antes. |
| previous_cancellations | Numérica | Cancelaciones previas. |
| previous_bookings_not_canceled | Numérica | Reservas previas no canceladas. |
| reserved_room_type | Categórica | Habitación reservada. |
| assigned_room_type | Categórica | Habitación asignada. |
| booking_changes | Numérica | Cambios realizados a la reserva. |
| deposit_type | Categórica | Política de depósito. |
| agent | Categórica (ID) | Código del agente. |
| company | Categórica (ID) | Código de la compañía. |
| days_in_waiting_list | Numérica | Días en lista de espera. |
| customer_type | Categórica | Tipo de cliente. |
| adr | Numérica | Tarifa promedio diaria. |
| required_car_parking_spaces | Numérica | Espacios de estacionamiento solicitados. |
| total_of_special_requests | Numérica | Cantidad de pedidos especiales. |

---


### Resumen descriptivo

In [7]:
df.describe().round(2)

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,21511.00,1426.00,25000.00,25000.00,25000.00,25000.00
mean,0.37,103.29,2024.16,26.51,15.83,0.92,2.48,1.85,0.10,0.01,0.03,0.08,0.13,0.22,86.05,189.79,2.40,101.86,0.06,0.57
std,0.48,106.59,0.71,13.40,8.81,0.99,1.88,0.58,0.39,0.12,0.18,0.79,1.43,0.63,110.38,132.42,17.56,48.03,0.24,0.80
min,0.00,0.00,2023.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,9.00,0.00,-6.38,0.00,0.00
25%,0.00,18.00,2024.00,16.00,8.00,0.00,1.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,9.00,62.00,0.00,69.50,0.00,0.00
50%,0.00,68.00,2024.00,27.00,16.00,1.00,2.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,14.00,174.00,0.00,95.00,0.00,0.00
75%,1.00,159.00,2025.00,37.00,24.00,2.00,3.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,229.00,277.00,0.00,126.00,0.00,1.00
max,1.00,629.00,2025.00,52.00,31.00,14.00,34.00,50.00,3.00,10.00,1.00,26.00,66.00,14.00,531.00,539.00,391.00,510.00,3.00,5.00


In [8]:
df['booking_id'].value_counts()

booking_id
HB-014269    1
HB-018087    1
HB-022870    1
HB-048154    1
HB-060351    1
            ..
HB-088693    1
HB-102568    1
HB-061499    1
HB-075900    1
HB-078303    1
Name: count, Length: 25000, dtype: int64

## Caracterización descriptiva inicial:


### 1. booking_id


In [9]:


print(df["booking_id"].describe())
ids_unicos = df["booking_id"].nunique()
faltantes = df["booking_id"].isnull().sum()

print("\n Descripción:")

print(f"\nCantidad de faltantes: {faltantes}")
print (f"\nSe trata de {ids_unicos} IDs unicas")

count         25000
unique        25000
top       HB-014269
freq              1
Name: booking_id, dtype: object

 Descripción:

Cantidad de faltantes: 0

Se trata de 25000 IDs unicas


**Descripción:**  
Es un identificador único, no aporta información estadística. Se usa solo para referencia.

**Cantidad y Faltantes:** 
Se cuenta con 25000 IDs unicas, sin ninguna faltante

---

### 2. hotel

In [10]:

print(df["hotel"].shape)

conteo_total = df["hotel"].count()

faltantes = df["hotel"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["hotel"].nunique()

conteo = df["hotel"].value_counts()
porcentaje = df["hotel"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)



(25000,)
Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
hotel
City Hotel      16716
Resort Hotel     8284
Name: count, dtype: int64

Porcentaje:
hotel
City Hotel      66.864
Resort Hotel    33.136
Name: proportion, dtype: float64


**Descripción:**  
La variable `hotel` indica el tipo de establecimiento donde se realizó la reserva. Es útil para comparar comportamientos entre City Hotel y Resort Hotel, ya que cada uno puede tener patrones diferentes de demanda, estacionalidad y cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2

**Distribución:**  
- City Hotel: 67%  
- Resort Hotel: 33%

**Observación:**  
La mayoría de las reservas corresponden al City Hotel. 

---

### 3. is_canceled

In [11]:
conteo_total = df["is_canceled"].count()

faltantes = df["is_canceled"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100

distintos = df["is_canceled"].nunique()

conteo = df["is_canceled"].value_counts()
porcentaje = df["is_canceled"].value_counts(normalize=True) * 100

print("Datos totales:", conteo_total)
print("Faltantes:", faltantes)
print("Porcentaje de faltantes:", porcentaje_faltantes, "%")

print("Cantidad de valores distintos:", distintos)

print("\nDistribución:")
print(conteo)

print("\nPorcentaje:")
print(porcentaje)




Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0 %
Cantidad de valores distintos: 2

Distribución:
is_canceled
0    15729
1     9271
Name: count, dtype: int64

Porcentaje:
is_canceled
0    62.916
1    37.084
Name: proportion, dtype: float64


### 3. is_canceled

**Descripción:**  
La variable `is_canceled` indica si la reserva fue cancelada (`1`) o no (`0`). Es una variable clave porque representa el resultado final del proceso de reserva y suele ser la variable objetivo en modelos de predicción de cancelaciones.

**Faltantes:**  
0 (0%)

**Valores distintos:**  
2  
- `0`: reserva no cancelada  
- `1`: reserva cancelada  

**Distribución:**  
- No canceladas: ~60%  
- Canceladas: ~40%

**Observación:**  
El dataset presenta una proporción considerable de cancelaciones. Esta distribución es importante para evaluar y llegar a una conclusion del motivo tan alto de las cancelaciones.

---

### 4. lead_time

In [12]:
conteo_total = df["lead_time"].count()
faltantes = df["lead_time"].isnull().sum()
porcentaje_faltantes = (faltantes / len(df)) * 100
distintos = df["lead_time"].nunique()

print("\nDatos totales:", conteo_total)
print(f"Faltantes: {faltantes}")
print(f"Porcentaje de faltantes: {porcentaje_faltantes}%")
print(f"Cantidad de valores distintos: {distintos}")


print("\nEstadisticas:")
print(df["lead_time"].describe())







Datos totales: 25000
Faltantes: 0
Porcentaje de faltantes: 0.0%
Cantidad de valores distintos: 460

Estadisticas:
count    25000.000000
mean       103.285280
std        106.585179
min          0.000000
25%         18.000000
50%         68.000000
75%        159.000000
max        629.000000
Name: lead_time, dtype: float64



**Descripción:**  
La variable lead_time representa la cantidad de días entre la fecha de reserva y la fecha de llegada (anticipacion). 

**Faltantes:**  
0 

Se trata de 460 valores diferentes. De un total de 25000 datos, sin ningun faltante.

**Hay una endencia central:** La media es 103 días, pero la mediana es 68, esto nos dice que la distribución está sesgada por valores extremos altos.

**Dispersión:** La desviación estándar es 106 días, bastante elevada, lo que confirma gran variabilidad.

**Rango:** Va de 0 (reservas hechas el mismo día) hasta 629 días (más de un año y medio de anticipación).

Con los **cuartiles** sabemos que:

  - 25% de las reservas se hacen con menos de 18 días de anticipación.

  - 50% con menos de 68 días.

  - 75% con menos de 159 días.

---


### 5. arrival_date_year


In [13]:
conteo_total = df["arrival_date_year"].count()

print("\nEstadisticas:")
print(df["arrival_date_year"].describe())

print("\nDatos y Cantidades:")
print(df["arrival_date_year"].value_counts())

print("\nPorcentajes:")
print(df["arrival_date_year"].value_counts(normalize=True))


Estadisticas:
count    25000.000000
mean      2024.158880
std          0.706072
min       2023.000000
25%       2024.000000
50%       2024.000000
75%       2025.000000
max       2025.000000
Name: arrival_date_year, dtype: float64

Datos y Cantidades:
arrival_date_year
2024    11906
2025     8533
2023     4561
Name: count, dtype: int64

Porcentajes:
arrival_date_year
2024    0.47624
2025    0.34132
2023    0.18244
Name: proportion, dtype: float64


**Descripcion:**
La variable `arrival_date_year` representa el año de llegada del huésped al hotel. 

**Faltantes:**
Está completa (0 faltantes) 

Contiene solo tres valores distintos: **2023**, **2024** y **2025**.

- El año 2024 concentra casi la mitad de las reservas (47,6 %), siendo el período con mas huespedes  del dataset.
- El año 2025 aporta un 34,1 %, mostrando continuidad hacia el futuro.
- El año 2023 tiene el 18,2 %, con menor peso porque es el inicio del registro.

Se deberia analizar el motivo por el cual en el 2024 hubo mayor cantidad de huespedes y esta bajo durante el 2025.

---

### 6. arrival_date_month 

In [14]:
print(f"Estadisticas basicas: ")
print(df["arrival_date_month"].describe())

print("\nCantidad por mes:")
print(df["arrival_date_month"].value_counts())

print("\nPorcentajes por mes:")
print(df["arrival_date_month"].value_counts(normalize=True).round(2))

print(f"\n Meses de los que tenemos datos: ")
print(df["arrival_date_month"].unique())


Estadisticas basicas: 
count      25000
unique        12
top       August
freq        2861
Name: arrival_date_month, dtype: object

Cantidad por mes:
arrival_date_month
August       2861
July         2691
May          2448
April        2379
October      2356
June         2321
September    2165
March        2020
February     1723
December     1389
November     1370
January      1277
Name: count, dtype: int64

Porcentajes por mes:
arrival_date_month
August       0.11
July         0.11
May          0.10
April        0.10
October      0.09
June         0.09
September    0.09
March        0.08
February     0.07
December     0.06
November     0.05
January      0.05
Name: proportion, dtype: float64

 Meses de los que tenemos datos: 
<StringArray>
[ 'February',  'November',     'April',     'March',    'August',       'May',
      'July', 'September',  'December',   'October',   'January',      'June']
Length: 12, dtype: str


**Descripción:**

La variable `arrival_date_month` representa el mes de llegada del huésped al hotel.
Faltantes:

**Faltantes:**
Está completa (0 faltantes).

Contiene 12 valores distintos, correspondientes a todos los meses del año.

- Los meses con mayor cantidad de huéspedes son Agosto (11%), Julio (11%) y Mayo (10%), mostrando la temporada alta.

- Meses como Enero (5%), Noviembre (5%) y Diciembre (6%) tienen un menor volumen de reservas.

Se debería analizar si esta distribucion de las reservas se relaciona directamente con estacionalidad o hay otros factores claves que influyan.

---

### 7. arrival_date_week_number

In [15]:

conteo_total = df["arrival_date_week_number"].count()
print("Conteo total:", conteo_total)

print("\nCantidad de valores distintos:")
print(f"{df["arrival_date_week_number"].nunique()} Semanas en el año")

print("\n Informacion:")
print(df["arrival_date_week_number"].info())

print("\nEstadísticas:")
print(df["arrival_date_week_number"].describe())

print("\nDatos y Cantidades:")
print(df["arrival_date_week_number"].value_counts().sort_index())

print("\nPorcentajes:")
print(df["arrival_date_week_number"].value_counts(normalize=True).round(4))




Conteo total: 25000

Cantidad de valores distintos:
52 Semanas en el año

 Informacion:
<class 'pandas.Series'>
RangeIndex: 25000 entries, 0 to 24999
Series name: arrival_date_week_number
Non-Null Count  Dtype
--------------  -----
25000 non-null  int64
dtypes: int64(1)
memory usage: 195.4 KB
None

Estadísticas:
count    25000.00000
mean        26.50996
std         13.39698
min          1.00000
25%         16.00000
50%         27.00000
75%         37.00000
max         52.00000
Name: arrival_date_week_number, dtype: float64

Datos y Cantidades:
arrival_date_week_number
1     343
2     200
3     323
4     338
5     300
6     323
7     462
8     436
9     549
10    385
11    454
12    511
13    439
14    485
15    538
16    552
17    594
18    598
19    516
20    591
21    609
22    513
23    581
24    550
25    522
26    550
27    578
28    614
29    634
30    612
31    594
32    703
33    735
34    589
35    578
36    471
37    527
38    562
39    471
40    571
41    565
42    533
43   

**Descripción:**

La variable `arrival_date_week_number` representa la semana del año en la que el huésped llega al hotel, con valores que van del 1 al 52.
Faltantes:

**Faltantes:**
Está completa (0 faltantes).

Contiene 52 valores distintos, cubriendo todas las semanas del año.

- La distribución es relativamente uniforme, con una media de 26.5 y una mediana de 27, lo que indica que las llegadas se concentran hacia la mitad del año.

- Las semanas con mayor cantidad de huéspedes se ubican entre la 27 y la 33, donde se observan los picos más altos (hasta 2.032 reservas).

- Las semanas iniciales y finales del año muestran menor actividad, especialmente la semana 1 y la semana 52, que presentan los valores más bajos del conjunto.

- La variable presenta baja asimetría (skew ≈ 0) y kurtosis negativa, indicando una distribución bastante plana y sin extremos marcados.

Como analisis preliminar se puede asumir que los picos en semanas centrales están asociados a vacaciones o temporadas turísticas específicas que incrementan la demanda.

---

### 8. arrival_date_day_of_month 

In [16]:
print(f"Conteo total: {df['arrival_date_day_of_month'].count()}")

print(f"\nCantidad de valores distintos: {df['arrival_date_day_of_month'].nunique()}")

print(f"\nNulos: {df['arrival_date_day_of_month'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['arrival_date_day_of_month'].describe()}")

print(f"\nDatos y Cantidades:\n{df['arrival_date_day_of_month'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['arrival_date_day_of_month'].value_counts(normalize=True).round(4)}")





Conteo total: 25000

Cantidad de valores distintos: 31

Nulos: 0

Estadísticas:
count    25000.000000
mean        15.830240
std          8.806273
min          1.000000
25%          8.000000
50%         16.000000
75%         24.000000
max         31.000000
Name: arrival_date_day_of_month, dtype: float64

Datos y Cantidades:
arrival_date_day_of_month
1     784
2     881
3     792
4     802
5     927
6     740
7     747
8     749
9     879
10    767
11    741
12    845
13    788
14    785
15    877
16    857
17    945
18    877
19    825
20    806
21    808
22    762
23    752
24    860
25    885
26    838
27    810
28    796
29    747
30    859
31    469
Name: count, dtype: int64

Porcentajes:
arrival_date_day_of_month
17    0.0378
5     0.0371
25    0.0354
2     0.0352
9     0.0352
18    0.0351
15    0.0351
24    0.0344
30    0.0344
16    0.0343
12    0.0338
26    0.0335
19    0.0330
27    0.0324
21    0.0323
20    0.0322
4     0.0321
28    0.0318
3     0.0317
13    0.0315
14    0.0314


**Descripción**

La variable `arrival_date_day_of_month` representa el día del mes en el que el huésped llega al hotel, con valores entre 1 y 31.

**Faltantes**

Está completa (0 nulos).

Contiene 31 valores diferentes, cubriendo todos los días posibles del mes.

Los estadísticos muestran una distribución equilibrada, lo que nos marca que las llegadas de los huespedes son estables y no hay concentraciones en dias particulares. 

**Frecuencias por día**

- El dia con mayor cantidad de huéspedes es el día 17 con 945

- El día con menor cantidad de huéspedes es el día 31 con 469

- Los días más frecuentes rondan entre 3.5% y 3.7% del total.

- El día 31 tiene el porcentaje más bajo (1.88%).


Las llegadas se concentran en días centrales del mes, mientras que los extremos muestran menor actividad. Esto sugiere un patrón estable de reservas que favorece la mitad del mes.

---

### 9. arrival_date

In [17]:
print(f"Conteo total: {df['arrival_date'].count()}")

print(f"Cantidad de valores distintos: {df['arrival_date'].nunique()}")

print(f"Nulos: {df['arrival_date'].isnull().sum()}")


#valor mas frecuente y porcentaje

valor_mas_frecuente = df['arrival_date'].mode()[0]
frecuencia = df['arrival_date'].value_counts()[valor_mas_frecuente]
porcentaje = frecuencia / df['arrival_date'].count() * 100

print(f"\nValor más frecuente: {valor_mas_frecuente}")
print(f"Frecuencia: {frecuencia}")
print(f"Porcentaje del valor mas frecuente: {porcentaje:.2f}%")



Conteo total: 25000
Cantidad de valores distintos: 793
Nulos: 0

Valor más frecuente: 2023-12-05
Frecuencia: 88
Porcentaje del valor mas frecuente: 0.35%


**Descripción**

La variable arrival_date representa la fecha completa de llegada del huésped al hotel.

**Faltantes**

Está completa (0 nulos).

**Valores**

Contiene 793 valores diferentes, lo que muestra una alta variabilidad en las fechas registradas.

- Valor más frecuente: 2023‑12‑05

- Frecuencia: 88

- Porcentaje: 0.35%

Las fechas de llegada están muy dispersas y no se concentran en días específicos. El valor más frecuente representa solo el 0.35% del total, lo que muestra mucha variabilidad sin ningun pico.

---

### 10. stays_in_weekend_nights

In [18]:
print(f"Conteo total: {df['stays_in_weekend_nights'].count()}")
print(f"Cantidad de valores distintos: {df['stays_in_weekend_nights'].nunique()}")
print(f"Nulos: {df['stays_in_weekend_nights'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['stays_in_weekend_nights'].describe()}")

print(f"\nDatos y Cantidades:\n{df['stays_in_weekend_nights'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['stays_in_weekend_nights'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['stays_in_weekend_nights'].min()}  |  Max: {df['stays_in_weekend_nights'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 12
Nulos: 0

Estadísticas:
count    25000.000000
mean         0.916640
std          0.991227
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         14.000000
Name: stays_in_weekend_nights, dtype: float64

Datos y Cantidades:
stays_in_weekend_nights
0     10983
1      6486
2      6817
3       248
4       388
5        19
6        37
7         4
8        13
9         3
10        1
14        1
Name: count, dtype: int64

Porcentajes:
stays_in_weekend_nights
0     0.4393
2     0.2727
1     0.2594
4     0.0155
3     0.0099
6     0.0015
5     0.0008
8     0.0005
7     0.0002
9     0.0001
14    0.0000
10    0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 14


**Descripción**

La variable `stays_in_weekend_nights` indica cuántas noches de fin de semana (viernes y sábado) permaneció el huésped en el hotel. Los valores observados van desde 0 hasta 14.

**Faltantes**
Está completa (0 nulos).

Contiene 12 valores diferentes, lo que muestra que la mayoría de los huéspedes tienen pocas noches de fin de semana registradas.

Los estadísticos muestran una distribución fuertemente concentrada en valores bajos, mostrando que los huéspedes no pasan noches de fin de semana en el hotel.

**Frecuencias por valor**

- El valor más frecuente es 0 noches, con 10.983 huéspedes.

- Los valores 1 y 2 noches también son comunes, con 6.486 y 6.817 respectivamente.

- A partir de 3 noches, las frecuencias caen.

**Porcentajes** 

- 0 noches representa el 43.93% del total.

- 2 noches: 27.27%

- 1 noche: 25.94%

- El resto de los valores tienen porcentajes menores al 2%, mostrando que estancias largas de fin de semana son muy poco frecuentes.

La mayoría de los huéspedes no se alojan durante fines de semana, y quienes lo hacen suelen quedarse solo 1 o 2 noches. Esto sugiere que el hotel recibe principalmente reservas de corta duración y con poca actividad en fines de semana.

---

### 11. stays_in_week_nights

In [19]:
print(f"Conteo total: {df['stays_in_week_nights'].count()}")
print(f"Cantidad de valores distintos: {df['stays_in_week_nights'].nunique()}")
print(f"Nulos: {df['stays_in_week_nights'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['stays_in_week_nights'].describe()}")

print(f"\nDatos y Cantidades:\n{df['stays_in_week_nights'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['stays_in_week_nights'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['stays_in_week_nights'].min()}  |  Max: {df['stays_in_week_nights'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 25
Nulos: 0

Estadísticas:
count    25000.000000
mean         2.479680
std          1.878629
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         34.000000
Name: stays_in_week_nights, dtype: float64

Datos y Cantidades:
stays_in_week_nights
0     1584
1     6368
2     7246
3     4587
4     1978
5     2262
6      313
7      192
8      121
9       44
10     214
11      17
12       9
13       3
14       5
15      22
16       7
17       1
18       1
19      11
20       4
21       5
22       3
24       2
34       1
Name: count, dtype: int64

Porcentajes:
stays_in_week_nights
2     0.2898
1     0.2547
3     0.1835
5     0.0905
4     0.0791
0     0.0634
6     0.0125
10    0.0086
7     0.0077
8     0.0048
9     0.0018
15    0.0009
11    0.0007
19    0.0004
12    0.0004
16    0.0003
21    0.0002
14    0.0002
20    0.0002
13    0.0001
22    0.0001
24    0.0001
18    0.0000
34    0.0000
17    0.0000


**Descripción**

La variable `stays_in_week_nights` representa la cantidad de noches de días de semana (lunes a jueves) que el  huésped permaneció en el hotel. Los valores observados van desde 0 hasta 34.

**Faltantes**

Está completa (0 nulos).

Contiene 25 valores diferentes, lo que muestra una mayor variabilidad respecto a las noches de fin de semana.

Los estadísticos muestran una distribución concentrada en valores bajos, mostrando estancias cortas durante días de semana.

**Frecuencias por valor**

- Los valores más frecuentes son 2 noches (28.98%), 1 noche (25.47%) y 3 noches (18.35%).

- A partir de 4 noches, las frecuencias disminuyen

- Los valores altos (17, 18, 24, 34) aparecen en cantidades mínimas, entre 1 y 2 registros.

**Porcentajes**

- Más del 70% de los huéspedes se alojan entre 1 y 3 noches.

- Las estadias de 4 a 6 noches representan porcentajes moderados.

- Las estadias largas (más de 10 noches) son extremadamente raras, con porcentajes cercanos a 0%.

La distribución muestra que los huéspedes suelen tener estadias cortas durante días de semana, con muy pocos casos de estadías prolongadas. 

---

### 12. Adults

In [20]:
print(f"Conteo total: {df['adults'].count()}")
print(f"Cantidad de valores distintos: {df['adults'].nunique()}")
print(f"Nulos: {df['adults'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['adults'].describe()}")

print(f"\nDatos y Cantidades:\n{df['adults'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['adults'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['adults'].min()}  |  Max: {df['adults'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 7
Nulos: 0

Estadísticas:
count    25000.000000
mean         1.853840
std          0.577313
min          0.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         50.000000
Name: adults, dtype: float64

Datos y Cantidades:
adults
0        80
1      4875
2     18727
3      1302
4        14
5         1
50        1
Name: count, dtype: int64

Porcentajes:
adults
2     0.7491
1     0.1950
3     0.0521
0     0.0032
4     0.0006
50    0.0000
5     0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 50


**Descripción**

La variable adults representa la cantidad de adultos incluidos en cada reserva. Los valores observados van desde 0 hasta 50, aunque la distribución muestra que casi todas las reservas contienen entre 1 y 3 adultos.

**Faltantes**

Está completa (0 nulos).

Contiene 7 valores diferentes, lo que indica una variabilidad muy baja en la cantidad de adultos por reserva.

Los estadísticos muestran una distribución fuertemente concentrada en valores bajos, donde la mayoria son reservas para 1 o 2 adultos.

**Frecuencias**

- 2 adultos es el valor más frecuente, con 18.727 registros (74.91%).

- 1 adulto aparece en 4.875 reservas (19.50%).

- 3 adultos representa 1.302 casos (5.21%).

- El valor máximo observado (50 adultos) es un outlier muy extremo.

**Porcentajes** 

- Más del 94% de las reservas tienen entre 1 y 2 adultos.

- Los valores superiores a 3 adultos representan porcentajes prácticamente nulos.

- La distribución presenta una asimetría extrema, con valores atípicos que no reflejan el comportamiento típico del dataset. (El outlier de 50 siendo un solo caso extremo)


La variable muestra que las reservas casi siempre incluyen 1 o 2 adultos, con muy pocos casos de grupos grandes. Los valores extremos (como 50 adultos) son outliers que no representan el patrón habitual de las reservas.

---

### 13. Children

In [21]:
print(f"Conteo total: {df['children'].count()}")
print(f"Cantidad de valores distintos: {df['children'].nunique()}")
print(f"Nulos: {df['children'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['children'].describe()}")

print(f"\nDatos y Cantidades:\n{df['children'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['children'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['children'].min()}  |  Max: {df['children'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 4
Nulos: 0

Estadísticas:
count    25000.000000
mean         0.101200
std          0.391005
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.000000
Name: children, dtype: float64

Datos y Cantidades:
children
0.0    23227
1.0     1033
2.0      723
3.0       17
Name: count, dtype: int64

Porcentajes:
children
0.0    0.9291
1.0    0.0413
2.0    0.0289
3.0    0.0007
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0.0  |  Max: 3.0


**Descripción**

La variable children representa la cantidad de niños incluidos en cada reserva. Los valores observados van desde 0 hasta 3, mostrando que la mayoría de las reservas no incluyen menores.

**Faltantes**

Está completa (0 nulos).

Contiene 4 valores diferentes, lo que indica una variabilidad muy baja en la cantidad de niños por reserva.

Los estadísticos muestran una distribución extremadamente concentrada en el valor 0, con muy pocos casos de reservas que incluyen niños.
Frecuencias por valor

- 0 niños es el valor dominante, con 23.227 registros.

- 1 niño aparece en 1.033 reservas.

- 2 niños aparece en 723 casos.

- 3 niños es muy raro, con solo 17 registros.

La distribución está fuertemente sesgada hacia reservas sin niños.

**Porcentajes** 

- 0 niños representa más del 92% del total.

- Los valores 1 y 2 niños representan porcentajes bajos, entre 3% y 4%.

- El valor 3 niños es prácticamente nulo en el dataset.

La mediana, el 25%, 50% y 75% percentil son todos 0, lo que confirma que la presencia de niños es poco frecuente.


La mayoría de las reservas no incluyen niños, y los casos con más de un menor son muy poco comunes. Esto sugiere que el hotel recibe principalmente huéspedes adultos o familias pequeñas con uno o dos niños como máximo.

---

### 14. Babies

In [22]:
print(f"Conteo total: {df['babies'].count()}")
print(f"Cantidad de valores distintos: {df['babies'].nunique()}")
print(f"Nulos: {df['babies'].isnull().sum()}")

print(f"\nEstadísticas:\n{df['babies'].describe()}")

print(f"\nDatos y Cantidades:\n{df['babies'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['babies'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['babies'].min()}  |  Max: {df['babies'].max()}")


Conteo total: 25000
Cantidad de valores distintos: 5
Nulos: 0

Estadísticas:
count    25000.000000
mean         0.007560
std          0.118926
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         10.000000
Name: babies, dtype: float64

Datos y Cantidades:
babies
0     24830
1       166
2         2
9         1
10        1
Name: count, dtype: int64

Porcentajes:
babies
0     0.9932
1     0.0066
2     0.0001
10    0.0000
9     0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 10


**Descripción**

La variable babies representa la cantidad de bebés incluidos en cada reserva. Los valores observados van desde 0 hasta 10, aunque la distribución muestra que prácticamente todas las reservas no incluyen bebés.

**Faltantes**

Está completa (0 nulos).

Contiene 5 valores diferentes, lo que muestra una variabilidad muy baja en la presencia de bebés dentro de las reservas.

Los estadísticos muestran una distribución concentrada en el valor 0, con valores máximos que corresponden a casos extremos atipicos.

**Frecuencias** 

- 0 bebés es el valor dominante, con 24.830 registros.

- 1 bebé aparece en 166 reservas.

- 2 bebés aparece en 2 casos.

- Los valores 9 y 10 bebés aparecen 1 vez cada uno, siendo outliers extremos.

La distribución está fuertemente sesgada hacia reservas sin bebés.

**Porcentajes** 

- 0 bebés representa el 99.32% del total.

- 1 bebé representa el 0.66%.

- El resto de los valores tienen porcentajes prácticamente nulos.

Los percentiles 25%, 50% y 75% son todos 0, confirmando que la presencia de bebés es excepcional.

La gran mayoría de las reservas no incluyen bebés, y los casos con más de uno son extremadamente raros. Esto sugiere que el hotel recibe principalmente huéspedes adultos o familias sin bebés, con muy pocos registros de grupos que viajen con menores de muy corta edad.

---

### 15. Meal

In [23]:
print(f"Conteo total: {df['meal'].count()}")
print(f"Cantidad de valores distintos: {df['meal'].nunique()}")
print(f"Nulos: {df['meal'].isnull().sum()}")

print(f"\nEstadísticas:")
print(f"Valor más frecuente: {df['meal'].mode()[0]}")
print(f"Frecuencia: {df['meal'].value_counts()[df['meal'].mode()[0]]}")

print(f"\nDatos y Cantidades:\n{df['meal'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['meal'].value_counts(normalize=True).round(4)}")


Conteo total: 25000
Cantidad de valores distintos: 5
Nulos: 0

Estadísticas:
Valor más frecuente: BB
Frecuencia: 19288

Datos y Cantidades:
meal
BB           19288
FB             147
HB            3073
SC            2264
Undefined      228
Name: count, dtype: int64

Porcentajes:
meal
BB           0.7715
HB           0.1229
SC           0.0906
Undefined    0.0091
FB           0.0059
Name: proportion, dtype: float64


**Descripción**

La variable meal indica el tipo de comidas asociado a cada reserva. Los valores corresponden a categorías estándar utilizadas en hoteles:

- BB - Bed & Breakfast (Alojamiento y Desayuno)

- HB - Half Board (Desayuno y Cena - Media pension)

- SC - Self Catering (Cocina incluida, el huesped puede prepar sus comidas)

- FB - Full Board (Desayuno, almuerzo y cena - Pension completa)

- Undefined - categoría no especificada

**Faltantes**

Está completa (0 nulos). Pero se tiene 228 valores como undefined.

Contiene 5 valores diferentes, lo que muestra una oferta tipica de servicios en un hotel (Siendo 4 lo tipico, sin contar los undefined)


La distribución está fuertemente concentrada en un único valor, lo que indica una preferencia  por parte de los huéspedes.

**Frecuencias** 

- BB es el valor dominante, con 19.288 registros.

**Porcentajes** 

- BB representa el 77.15% del total.

La categoría BB domina ampliamente, mientras que las demás tienen una presencia mucho menor.

La mayoría de los huéspedes elige el régimen Bed & Breakfast, lo que sugiere que el hotel funciona principalmente con clientes que prefieren desayunar en el establecimiento y resolver el resto de las comidas por su cuenta. Los regímenes más completos (HB, FB) son minoritarios. 

---

### 16. Country

In [24]:
print(df['country'].describe())

print(f"\nNulos: {df['country'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['country'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['country'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['country'].value_counts(normalize=True).round(4)}")


count     24899
unique      129
top         PRT
freq      10278
Name: country, dtype: object

Nulos: 101
Cantidad de valores distintos: 129

Datos y Cantidades:
country
ABW     1
AGO    88
ALB     3
AND     2
ARE     6
       ..
VEN     9
VGB     1
VNM     2
ZAF    15
ZWE     1
Name: count, Length: 129, dtype: int64

Porcentajes:
country
PRT    0.4128
GBR    0.0989
FRA    0.0875
ESP    0.0714
DEU    0.0610
        ...  
SLE    0.0000
BEN    0.0000
GLP    0.0000
SYC    0.0000
TZA    0.0000
Name: proportion, Length: 129, dtype: float64


**Descripción**

La variable country identifica el país de origen del huésped. Es una variable categórica con alta diversidad,con 129 países distintos, lo que muestra una clientela internacional.

**Faltantes**
Presenta 101 valores faltantes, menos del 1%, por lo que la variable está prácticamente completa.

**Frecuencias** 

El país más frecuente es PRT, con una frecuencia muy superior al resto.

- PRT (Portugal) es el valor que mas aparece con 10.278 registros.

**Porcentajes**

- PRT (Portugal) representa el 41.28% del total.

- GBR (Reino Unido) esta en el 9.89% de las reservas.

- FRA (Francia) 8.75%.

- ESP (España) 7.14%.

- DEU (Alemania) 6.10%.

- Más del 40% de los registros se distribuyen entre países con porcentajes menores al 1% cada uno.

La mayoría de los huéspedes provienen de Portugal, seguido por Reino Unido, Francia, España y Alemania.
Aunque aparecen 129 países, la mayoría aporta muy pocos registros, lo que muestra la presencia de huespedes internacionales pero la mayoria siendo de un grupo pequeño de paises

---

### 17. Market Segment

In [25]:
print(df['market_segment'].describe())

print(f"\nNulos: {df['market_segment'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['market_segment'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['market_segment'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['market_segment'].value_counts(normalize=True).round(4)}")

count         25000
unique            7
top       Online TA
freq          11867
Name: market_segment, dtype: object

Nulos: 0
Cantidad de valores distintos: 7

Datos y Cantidades:
market_segment
Aviation            47
Complementary      148
Corporate         1114
Direct            2650
Groups            4163
Offline TA/TO     5011
Online TA        11867
Name: count, dtype: int64

Porcentajes:
market_segment
Online TA        0.4747
Offline TA/TO    0.2004
Groups           0.1665
Direct           0.1060
Corporate        0.0446
Complementary    0.0059
Aviation         0.0019
Name: proportion, dtype: float64


**Descripción**

La variable market_segment indica el canal por el cual llega la reserva. 

**Faltantes**

No tiene valores faltantes y cuenta con 7 categorías distintas.

La mas repetida es Online TA con 11867 registros.

**Distribución** 

- Online TA: 47.47%

- Offline TA/TO: 20.04%

- Groups: 16.65%

- Direct: 10.60%

- Corporate: 4.46%

- Complementary: 0.59%

- Aviation: 0.19%

El valor más frecuente es Online TA, con 11.867 reservas.

La mayoría de las reservas provienen de agencias online, mostrando una fuerte dependencia de canales digitales. Los segmentos offline, grupos y directos también aportan volumen relevante.

---

### 18. distribution channel 

In [26]:
print(df['distribution_channel'].describe())

print(f"\nNulos: {df['distribution_channel'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['distribution_channel'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['distribution_channel'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['distribution_channel'].value_counts(normalize=True).round(4)}")


count     25000
unique        4
top       TA/TO
freq      20488
Name: distribution_channel, dtype: object

Nulos: 0
Cantidad de valores distintos: 4

Datos y Cantidades:
distribution_channel
Corporate     1409
Direct        3063
GDS             40
TA/TO        20488
Name: count, dtype: int64

Porcentajes:
distribution_channel
TA/TO        0.8195
Direct       0.1225
Corporate    0.0564
GDS          0.0016
Name: proportion, dtype: float64


**Descripción**

La variable indica el canal por el cual se distribuye la reserva. 



**Faltantes**
No presenta valores faltantes y contiene 4 categorías, lo que muestra un esquema de distribución simple.

- Travel Agent (Agencia de Viajes) / Tour Operator (Operador Turístico)
- Direct (web del hotel / teléfono)
- Corporate (acuerdos corporativos)
- GDS (sistemas globales de distribución)

**Distribución y porcentajes** 

- TA/TO: 20488 (81.95%)

- Direct: 3063 (12.25%)

- Corporate: 1409 (5.64%)

- GDS: 40 (0.16%)

El valor más frecuente es TA/TO, con 20.488 registros.

La gran mayoría de las reservas se gestionan a través de agencias y operadores turísticos, lo que muestra una fuerte dependencia de intermediarios para captar clientes. Los canales directos y corporativos aportan una proporción menor pero no inexistente, mostrando que el hotel también tiene sus ventas directas y acuerdos empresariales. 

---

### 19. Is repeated guest

In [27]:
print(df['is_repeated_guest'].describe())

print(f"\nNulos: {df['is_repeated_guest'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['is_repeated_guest'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['is_repeated_guest'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['is_repeated_guest'].value_counts(normalize=True).round(4)}")


count    25000.000000
mean         0.032600
std          0.177591
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: is_repeated_guest, dtype: float64

Nulos: 0
Cantidad de valores distintos: 2

Datos y Cantidades:
is_repeated_guest
0    24185
1      815
Name: count, dtype: int64

Porcentajes:
is_repeated_guest
0    0.9674
1    0.0326
Name: proportion, dtype: float64


**Descripción**

La variable indica si el huésped es recurrente (1) o nuevo (0). 

**Faltantes**

No presenta valores faltantes y contiene únicamente 2 categorías, funcionando como un booleano.

**Distribución** 

- 0 (no repetido): 24.185 registros (96.74%)

- 1 (repetido): 815 registros (3.26%)

Los estadísticos muestran que la mayoría de los valores son 0, con percentiles 25%, 50% y 75% todos en 0, lo que confirma una distribución concentrada en huéspedes nuevos.


El hotel recibe principalmente huéspedes nuevos, lo que sugiere una tasa baja de repetición. 

---

### 20. Previous Cancellations

In [28]:
print(df['previous_cancellations'].describe())

print(f"\nNulos: {df['previous_cancellations'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['previous_cancellations'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['previous_cancellations'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['previous_cancellations'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['previous_cancellations'].min()}  |  Max: {df['previous_cancellations'].max()}")


count    25000.000000
mean         0.084080
std          0.791148
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         26.000000
Name: previous_cancellations, dtype: float64

Nulos: 0
Cantidad de valores distintos: 14

Datos y Cantidades:
previous_cancellations
0     23633
1      1283
2        21
3        14
4         8
5         2
6         4
11        8
13        3
14        2
19        5
24        9
25        5
26        3
Name: count, dtype: int64

Porcentajes:
previous_cancellations
0     0.9453
1     0.0513
2     0.0008
3     0.0006
24    0.0004
4     0.0003
11    0.0003
25    0.0002
19    0.0002
6     0.0002
13    0.0001
26    0.0001
5     0.0001
14    0.0001
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 26


**Descripción**

La variable indica cuántas reservas anteriores fueron canceladas por el mismo huésped antes de la actual. 

**Faltantes**

No presenta valores faltantes 

**Estadisticas**

Contiene 14 valores distintos, con un rango que va desde 0 hasta 26. 
Los estadísticos muestran que los percentiles 25%, 50% y 75% son todos 0, lo que muestra una distribución concentrada en ese valor.

**Distribución y porcentajes** 

- 0 cancelaciones previas: 23.633 registros (94.53%)

- 1 cancelación previa: 1.283 registros (5.13%)

- Valores mayores a 1 : proporciones menores al 1%, con frecuencias muy bajas 

- Casos extremos: valores como 24, 25 y 26 aparecen, pero representan proporciones menores al 0.04%. (Outliers)


La enorme concentración en 0 cancelaciones previas indica que la mayoría de los huéspedes no tiene historial de cancelaciones, lo que se entiende tendiendo en cuenta que la gran mayoria de los clientes son clientes nuevos. Los valores altos (hasta 26) son casos excepcionales.

---


### 21. Previous bookings not canceled

In [29]:
print(df['previous_bookings_not_canceled'].describe())

print(f"\nNulos: {df['previous_bookings_not_canceled'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['previous_bookings_not_canceled'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['previous_bookings_not_canceled'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['previous_bookings_not_canceled'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['previous_bookings_not_canceled'].min()}  |  Max: {df['previous_bookings_not_canceled'].max()}")


count    25000.000000
mean         0.133840
std          1.425766
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         66.000000
Name: previous_bookings_not_canceled, dtype: float64

Nulos: 0
Cantidad de valores distintos: 38

Datos y Cantidades:
previous_bookings_not_canceled
0     24207
1       344
2       130
3        80
4        54
5        41
6        18
7        15
8        12
9        13
10       12
11       12
12        5
13        9
14        5
15        5
16        4
17        2
18        2
19        2
20        2
21        2
22        2
23        1
24        2
25        4
26        1
27        4
31        1
37        1
38        1
40        1
44        1
45        1
54        1
58        1
65        1
66        1
Name: count, dtype: int64

Porcentajes:
previous_bookings_not_canceled
0     0.9683
1     0.0138
2     0.0052
3     0.0032
4     0.0022
5     0.0016
6     0.0007
7     0.0006
9     0.0005
10    0.0005
11    0.0005
8    

**Descripción**

La variable indica cuántas reservas anteriores no canceladas realizó el huésped antes de la actual. 

**Faltantes**

No tiene valores faltantes 

**Estadisticas**

Contiene 38 valores distintos, con un rango que va desde 0 hasta 66.

Los percentiles 25%, 50% y 75% son 0, lo que dice que la distribución está concentrada en huéspedes sin historial previo de reservas.
Distribución principal

**Distribución y porcentajes**

- 0 reservas previas no canceladas: 24.207 registros (96.83%)

- 1 reserva previa: 344 registros (1.38%)

- 2 a 5 reservas previas: proporciones entre 0.16% y 0.52%

- Valores mayores a 10: extremadamente raros (menos de 0.05%)

- Casos extremos: aparecen valores como 54, 58, 65 y 66, cada uno con solo 1 registro.


La enorme concentración en 0 reservas previas no canceladas indica que la mayoría de los huéspedes no tiene historial de reservas exitosas anteriores, lo que corresponde con la informacion que se tiene de que la mayoria de los clientes son nuevos.
Los valores altos (más de 20 reservas previas) son casos excepcionales y pueden corresponder a clientes muy frecuentes.

---


### 22. Reserved room type

In [30]:

print(df['reserved_room_type'].describe())

print(f"\nNulos: {df['reserved_room_type'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['reserved_room_type'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['reserved_room_type'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['reserved_room_type'].value_counts(normalize=True).round(4)}")


count     25000
unique       10
top           A
freq      18112
Name: reserved_room_type, dtype: object

Nulos: 0
Cantidad de valores distintos: 10

Datos y Cantidades:
reserved_room_type
A    18112
B      209
C      187
D     3963
E     1357
F      610
G      439
H      120
L        1
P        2
Name: count, dtype: int64

Porcentajes:
reserved_room_type
A    0.7245
D    0.1585
E    0.0543
F    0.0244
G    0.0176
B    0.0084
C    0.0075
H    0.0048
P    0.0001
L    0.0000
Name: proportion, dtype: float64


**Descripción**

Variable categórica que indica el tipo de habitación reservada por los huespedes.

**Faltantes**

No tiene faltantes

**Estadisticas**

Cuenta con 10 categorías, no se cuenta con mas informacion acerca del tipo de habitacion o costo. Simplemente el tipo.

**Distribucion y porcentajes**

- A: 18112 (72.45%)

- D: 3963 (15.85%)

- E: 1357 (5.43%)

- F: 610 (2.44%)

- G: 439 (1.76%)

La categoría A domina y representa la habitación estándar o más demandada. 

---

### 23. Assigned room type

In [34]:
print(df['assigned_room_type'].describe())

print(f"\nNulos: {df['assigned_room_type'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['assigned_room_type'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['assigned_room_type'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['assigned_room_type'].value_counts(normalize=True).round(4)}")


count     25000
unique       11
top           A
freq      15641
Name: assigned_room_type, dtype: object

Nulos: 0
Cantidad de valores distintos: 11

Datos y Cantidades:
assigned_room_type
A    15641
B      436
C      488
D     5248
E     1606
F      796
G      527
H      135
I       65
K       56
P        2
Name: count, dtype: int64

Porcentajes:
assigned_room_type
A    0.6256
D    0.2099
E    0.0642
F    0.0318
G    0.0211
C    0.0195
B    0.0174
H    0.0054
I    0.0026
K    0.0022
P    0.0001
Name: proportion, dtype: float64


**Descripción**

La variable indica el tipo de habitación finalmente asignada al huésped.

**Faltantes**
 No presenta valores faltantes.
 
Contiene 11 categorías, lo que muestra una oferta grande en tipo de habitaciones.
El valor más frecuente es A, lo que confirma que es la habitación estándar o más disponible en el hotel.

**Distribución y porcentajes**

- A: 15.641 registros (62.56%)

- D: 5.248 registros (20.99%)

- E: 1.606 registros (6.42%)

- F: 796 registros (3.18%)

- G: 527 registros (2.11%)

- El resto de las categorías tienen proporciones menores al 2%

La distribución está concentrada en A y D, que juntas representan más del 83% de las asignaciones. Lo que coincide con el tipo de habitacion reservada.

La categoría A domina la asignación, lo que sugiere que es la habitación más abundante o la opción por defecto del hotel.
Las categorías con baja frecuencia (como I, K y P) probablemente corresponden a habitaciones especiales, con disponibilidad limitada o asignadas solo en casos puntuales.

---

### 24. booking changes

In [37]:
print(df['booking_changes'].describe())

print(f"\nNulos: {df['booking_changes'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['booking_changes'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['booking_changes'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['booking_changes'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['booking_changes'].min()}  |  Max: {df['booking_changes'].max()}")

count    25000.000000
mean         0.222040
std          0.633431
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         14.000000
Name: booking_changes, dtype: float64

Nulos: 0
Cantidad de valores distintos: 13

Datos y Cantidades:
booking_changes
0     21194
1      2647
2       813
3       225
4        69
5        22
6        15
7         7
8         4
9         1
10        1
13        1
14        1
Name: count, dtype: int64

Porcentajes:
booking_changes
0     0.8478
1     0.1059
2     0.0325
3     0.0090
4     0.0028
5     0.0009
6     0.0006
7     0.0003
8     0.0002
9     0.0000
13    0.0000
10    0.0000
14    0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 14


**Descripción**

Variable numérica que indica cuántas veces se modificó la reserva antes de su llegada (cambios de fechas, habitación, etc.)

**Faltantes**
No presenta valores faltantes


Contiene 13 valores distintos, con un rango entre 0 y 14.
Los percentiles 25%, 50% y 75% son 0, lo que confirma que la mayoría de las reservas no tuvo cambios.

**Distribución**

- 0 cambios: mayoría absoluta (más de 21.000 registros)

- 1 cambio: proporción baja

- 2 a 4 cambios: muy bajos

- 5 o mas cambios: extremadamente raros

- Máximo observado: 14 cambios (Outlier extremo)

La distribución está fuertemente concentrada en 0, lo que indica que la gran mayoría de las reservas se mantuvo sin modificaciones.

El hecho de que más del 80% de las reservas no tenga cambios sugiere que los huéspedes suelen mantener su reserva original sin modificaciones.

Esta variable es útil pporque un número alto de cambios puede relacionarse con mayor probabilidad de cancelación.

---

### 25. Deposit type

In [38]:
print(df['deposit_type'].describe())

print(f"\nNulos: {df['deposit_type'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['deposit_type'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['deposit_type'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['deposit_type'].value_counts(normalize=True).round(4)}")


count          25000
unique             3
top       No Deposit
freq           21836
Name: deposit_type, dtype: object

Nulos: 0
Cantidad de valores distintos: 3

Datos y Cantidades:
deposit_type
No Deposit    21836
Non Refund     3134
Refundable       30
Name: count, dtype: int64

Porcentajes:
deposit_type
No Deposit    0.8734
Non Refund    0.1254
Refundable    0.0012
Name: proportion, dtype: float64


**Descripción**

Variable categórica que indica el tipo de depósito asociado a la reserva.
Tiene 3 categorías: No Deposit (Sin deposito), Non Refund (No reembolsable), Refundable (Reembolsable).

**Faltantes**
No tiene faltantes faltantes

**Distribución y porcentaje**

- No Deposit: 21836 (87%)

- Non Refund: 3134 (13%)

- Refundable:30 (menos del 1%)

La distribución está fuertemente concentrada en No Deposit, lo que indica que la mayoría de las reservas no requieren pago anticipado.
Interpretación breve

La predominancia de No Deposit sugiere que el hotel opera principalmente con reservas sin compromiso previo, podria tratarse de una de las causas de la tasa tan altas de cancelaciones

---


### 26. Agent

In [39]:
print(df['agent'].describe())

print(f"\nNulos: {df['agent'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['agent'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['agent'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['agent'].value_counts(normalize=True).round(4)}")


count    21511.000000
mean        86.049835
std        110.384758
min          1.000000
25%          9.000000
50%         14.000000
75%        229.000000
max        531.000000
Name: agent, dtype: float64

Nulos: 3489
Cantidad de valores distintos: 268

Datos y Cantidades:
agent
1.0      1508
2.0        27
3.0       284
4.0         9
5.0        81
         ... 
502.0       5
509.0       1
526.0       4
527.0       6
531.0       9
Name: count, Length: 268, dtype: int64

Porcentajes:
agent
9.0      0.3110
240.0    0.1346
1.0      0.0701
7.0      0.0361
14.0     0.0351
          ...  
244.0    0.0000
291.0    0.0000
429.0    0.0000
258.0    0.0000
141.0    0.0000
Name: proportion, Length: 268, dtype: float64


**Descripción**

Variable numérica que identifica el ID del agente o agencia de viajes que gestionó la reserva.

**Faltantes**
Tiene 3489 valores nulos (14%), lo cual puede indicar reservas hechas sin agente (directas).

**Estadisticas**
Tiene 268 valores distintos, cuenta con una alta diversidad de agencias.

Los estadísticos muestran una distribución muy dispersa:

- Mínimo: 1

- Máximo: 531

- Mediana: 14

- Percentil 75: 229 

- Esto indica que hay muchos agentes con pocos registros y unos pocos con volúmenes muy altos.

**Distribución y porcentajes**

Los agentes más frecuentes son:

- 9: 31.10%

- 240: 13.46%

- 1: 7.01%

- 7: 3.61%

- 14: 3.51%

El resto de los agentes tiene proporciones muy bajas, muchos con menos de 0.1%.

La variable agent muestra que una parte importante de las reservas se realiza sin intermediarios (14% nulos), mientras que el resto se distribuye entre muchos agentes distintos, aunque unos pocos concentran la mayoría de las operaciones.

---

### 27. Company

In [40]:
print(df['company'].describe())

print(f"\nNulos: {df['company'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['company'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['company'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['company'].value_counts(normalize=True).round(4)}")


count    1426.000000
mean      189.791725
std       132.421976
min         9.000000
25%        62.000000
50%       174.000000
75%       277.000000
max       539.000000
Name: company, dtype: float64

Nulos: 23574
Cantidad de valores distintos: 218

Datos y Cantidades:
company
9.0      8
11.0     1
12.0     5
14.0     1
18.0     1
        ..
521.0    1
523.0    7
525.0    4
528.0    2
539.0    1
Name: count, Length: 218, dtype: int64

Porcentajes:
company
40.0     0.1424
223.0    0.1087
67.0     0.0372
45.0     0.0358
153.0    0.0288
          ...  
253.0    0.0007
260.0    0.0007
358.0    0.0007
160.0    0.0007
384.0    0.0007
Name: proportion, Length: 218, dtype: float64


**Descripción**
Variable que identifica la empresa asociada a la reserva.

**Faltantes**
Tiene 94% de valores nulos, lo que indica que casi todas las reservas no provienen de compañías sino de clientes individuales o agencias.

**Distribución**

Los valores no nulos son pocos (solo 1.426 registros) y están muy dispersos entre 218 empresas distintas, la mayoría con menos de 10 reservas.

Los IDs más frecuentes:

- 40: 14.24%

- 223: 10.87%

- 67: 3.72%

- 45: 3.58%

- 153: 2.88%

Las reservas corporativas son muy poco comunes en este dataset.
El hotel opera casi exclusivamente con clientes individuales, y las pocas empresas que aparecen lo hacen con volúmenes pequeños.

---

### 28. Days in waiting list.

In [42]:
print(df['days_in_waiting_list'].describe())

print(f"\nNulos: {df['days_in_waiting_list'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['days_in_waiting_list'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['days_in_waiting_list'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['days_in_waiting_list'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['days_in_waiting_list'].min()}  |  Max: {df['days_in_waiting_list'].max()}")

count    25000.000000
mean         2.400440
std         17.555068
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        391.000000
Name: days_in_waiting_list, dtype: float64

Nulos: 0
Cantidad de valores distintos: 100

Datos y Cantidades:
days_in_waiting_list
0      24174
1          4
2          1
3         13
4          5
       ...  
236        6
259        2
330        5
379        3
391        8
Name: count, Length: 100, dtype: int64

Porcentajes:
days_in_waiting_list
0      0.9670
39     0.0020
58     0.0016
31     0.0014
44     0.0011
        ...  
74     0.0000
100    0.0000
53     0.0000
13     0.0000
97     0.0000
Name: proportion, Length: 100, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 391


**Descripción**

Variable numérica que indica cuántos días estuvo una reserva en lista de espera antes de ser confirmada.

**Faltantes**

No tiene valores faltantes.

**Estadistica**

Contiene 100 valores distintos, con un rango entre 0 y 391 días.

**Distribución y porcentajes**

La variable esta completa en su mayoria por el valor 0, con 24.174 registros (96.7%).
El resto de los valores aparecen en proporciones mínimas, muchos con apenas 1 a 8 registros.
Los valores altos (desde 200 hasta 391) son casos extremadamente raros.

Percentiles:

- 25%: 0

- 50%: 0

- 75%: 0

Esto confirma que la gran mayoría de las reservas nunca estuvo en lista de espera.

El hotel casi no utiliza lista de espera: solo un 3.3% de las reservas pasó por este proceso, y en la mayoría de esos casos fueron esperas muy cortas. Los valores altos representan situaciones excepcionales, probablemente relacionadas con reservas especiales.

---


### 29. Customer type

In [43]:
print(df['customer_type'].describe())

print(f"\nNulos: {df['customer_type'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['customer_type'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['customer_type'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['customer_type'].value_counts(normalize=True).round(4)}")


count         25000
unique            4
top       Transient
freq          18788
Name: customer_type, dtype: object

Nulos: 0
Cantidad de valores distintos: 4

Datos y Cantidades:
customer_type
Contract             867
Group                118
Transient          18788
Transient-Party     5227
Name: count, dtype: int64

Porcentajes:
customer_type
Transient          0.7515
Transient-Party    0.2091
Contract           0.0347
Group              0.0047
Name: proportion, dtype: float64


**Descripción**

customer_type clasifica el tipo de cliente según el patrón de reserva. 

**Faltantes**

No tiene valores faltantes

**Categorias**

Se trata de 4 categorías definidas: Transient, Transient-Party, Contract y Group.

La categoría dominante es Transient, que representa clientes individuales que reservan sin formar parte de un grupo ni contrato.
La segunda categoría, Transient-Party, corresponde a clientes que viajan en pequeños grupos informales.
Las reservas Contract son minoritarias y suelen estar asociadas a convenios empresariales.
La categoría Group son clientes que forman parte de un grupo organizado.

**Distribución y porcentajes**

- Transient: 18788 (75.15%)

- Transient-Party: 5227 (20.91%)

- Contract : 867 (3.47%)

- Group: 118 (0.47%)

El hotel trabaja principalmente con clientes individuales, seguido por pequeños grupos informales. Las reservas corporativas y los grupos organizados son marginales. Esto sugiere un perfil de operación centrado en turismo general y viajeros independientes, con poca presencia de contratos empresariales o grandes grupos.

---

### 30. adr

In [45]:
print(df['adr'].describe())

print(f"\nNulos: {df['adr'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['adr'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['adr'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['adr'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['adr'].min()}  |  Max: {df['adr'].max()}")


count    25000.000000
mean       101.858566
std         48.030925
min         -6.380000
25%         69.500000
50%         95.000000
75%        126.000000
max        510.000000
Name: adr, dtype: float64

Nulos: 0
Cantidad de valores distintos: 3862

Datos y Cantidades:
adr
-6.38        1
 0.00      405
 0.50        1
 1.00        3
 1.29        1
          ... 
 382.00      1
 388.00      1
 392.00      1
 450.00      1
 510.00      1
Name: count, Length: 3862, dtype: int64

Porcentajes:
adr
62.00     0.0318
75.00     0.0228
65.00     0.0215
90.00     0.0205
0.00      0.0162
           ...  
48.57     0.0000
71.72     0.0000
43.33     0.0000
57.40     0.0000
169.75    0.0000
Name: proportion, Length: 3862, dtype: float64

Minimo y Maximo:
Min: -6.38  |  Max: 510.0


**Descripción** 

adr (Average Daily Rate) representa la tarifa diaria promedio pagada por el huésped.

**Faltantes**
No tiene valores faltantes 

**Distribución**
Tiene una variabilidad muy alta, con 3862 valores distintos sobre 25.000 registros.

- Media: 101.86

- Mediana: 95

- Rango intercuartílico: 69.5 a 126

- Mínimo: −6.38

- Máximo: 510

La distribución está concentrada entre 70 y 130, que es el rango típico de tarifas.
Los valores cercanos a 0 suelen ser reservas sin cargo o ajustes contables.

**Outliers**

Hay dos tipos claros de outliers:

*Outliers negativos*

- 6.38: aparece solo 1 vez.

Este valor es inválido para una tarifa y puede representar error de carga o una reversión de un cobro.

*Outliers positivos*

- Valores muy altos desde 300 hasta 510 aparecen en cantidades mínimas (1 registro cada uno).

Estos representan suites premium, tarifas corporativas especiales o posibles errores de carga.


La variable adr muestra que el hotel trabaja con tarifas promedio alrededor de 100 USD (asumiendo que la tarifa es en dolares) por noche, con una distribución estable y concentrada.
Los outliers negativos y los valores extremadamente altos deben revisarse porque pueden distorsionar los análisis.

---

### 31. Required car parking spaces

In [46]:
print(df['required_car_parking_spaces'].describe())

print(f"\nNulos: {df['required_car_parking_spaces'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['required_car_parking_spaces'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['required_car_parking_spaces'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['required_car_parking_spaces'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['required_car_parking_spaces'].min()}  |  Max: {df['required_car_parking_spaces'].max()}")


count    25000.000000
mean         0.060600
std          0.240437
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.000000
Name: required_car_parking_spaces, dtype: float64

Nulos: 0
Cantidad de valores distintos: 4

Datos y Cantidades:
required_car_parking_spaces
0    23495
1     1496
2        8
3        1
Name: count, dtype: int64

Porcentajes:
required_car_parking_spaces
0    0.9398
1    0.0598
2    0.0003
3    0.0000
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 3


**Descripción**

Esta variable indica cuántos espacios de estacionamiento solicitó el huésped al momento de la reserva.

**Faltantes**
No tiene valores faltantes 

Presenta solo 4 valores posibles: 0, 1, 2 y 3.

**Distribución y porcentajes**

- 0 espacios: 23495 (93.98%)

- 1 espacio: 1496 (5.98%)

- 2 espacios: 8 (0.03%)

- 3 espacios: 1 (0.004%)

La distribución está completamente dominada por el valor 0, lo que indica que la gran mayoría de los huéspedes no requiere estacionamiento.

**Percentiles:**

- 25%: 0

- 50%: 0

- 75%: 0

Esto confirma que el estacionamiento no es una necesidad proritaria. 


La variable muestra que el estacionamiento casi no es un factor relevante en las reservas: solo un pequeño porcentaje de huéspedes solicita un espacio, y los casos de 2 o 3 espacios son extremadamente raros. Esto sugiere que el hotel recibe principalmente huéspedes sin vehículo o que no necesitan estacionamiento durante su estadía.

---

### 32. Total of special requests

In [47]:
print(df['total_of_special_requests'].describe())

print(f"\nNulos: {df['total_of_special_requests'].isnull().sum()}")
print(f"Cantidad de valores distintos: {df['total_of_special_requests'].nunique()}")

print(f"\nDatos y Cantidades:\n{df['total_of_special_requests'].value_counts().sort_index()}")

print(f"\nPorcentajes:\n{df['total_of_special_requests'].value_counts(normalize=True).round(4)}")

print(f"\nMinimo y Maximo:\nMin: {df['total_of_special_requests'].min()}  |  Max: {df['total_of_special_requests'].max()}")


count    25000.000000
mean         0.573560
std          0.797229
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          5.000000
Name: total_of_special_requests, dtype: float64

Nulos: 0
Cantidad de valores distintos: 6

Datos y Cantidades:
total_of_special_requests
0    14712
1     6957
2     2714
3      527
4       77
5       13
Name: count, dtype: int64

Porcentajes:
total_of_special_requests
0    0.5885
1    0.2783
2    0.1086
3    0.0211
4    0.0031
5    0.0005
Name: proportion, dtype: float64

Minimo y Maximo:
Min: 0  |  Max: 5


**Descripción**

Variable que indica cuántas solicitudes especiales realizó el huésped al momento de la reserva.

**Faltantes**
No tiene valores faltantes.

Presenta 6 valores posibles: de 0 a 5.

**Distribución y porcentajes**

- 0 solicitudes: 14712 (58.85%)

- 1 solicitud: 6957 (27.83%)

- 2 solicitudes: 2714 (10.86%)

- 3 solicitudes: 527 (2.11%)

- 4 solicitudes: 77 (0.31%)

- 5 solicitudes: 13 (0.05%)

La mayoría de los huéspedes no realiza solicitudes especiales, y los que sí lo hacen generalmente piden 1 o 2.
Los valores altos (4 y 5) son extremadamente raros.

**Percentiles:**

- 25%: 0

- 50%: 0

- 75%: 1

Esto confirma que las solicitudes especiales son poco frecuentes y de baja complejidad.


La variable muestra que la mayoría de los huéspedes no requiere condiciones especiales en su estadía.